In [5]:
import pandas as pd

synonyms_file = r'C:\Users\Nattapot\Desktop\thesis_ef\extract_data\synonyms.txt'
with open(synonyms_file, encoding="utf-8") as f:
    synonyms_list = [line.strip().split(", ") for line in f.readlines()]

dataset_file = r'C:\Users\Nattapot\Desktop\thesis_ef\evaluate\emission_factor_20250324.csv'
df = pd.read_csv(dataset_file, header=None) 
df['combined'] = df[[0, 1, 2]].apply(lambda row: " ".join(str(value) for value in row if pd.notnull(value)), axis=1)

results = []
for synonyms in synonyms_list:
    keyword = synonyms[0] 
    matched_rows = []
    for index, row in df.iterrows():
        row_text = row['combined']
        if any(word in row_text for word in synonyms):
            matched_rows.append(index) 

    results.append({"Keyword": keyword, "Synonyms": ", ".join(synonyms), "Rows Found": matched_rows})
result_df = pd.DataFrame(results)

result_df.to_csv("ground_truth_hybrid.csv",index=False,encoding='utf-8-sig')

In [6]:
gt = r'C:\Users\Nattapot\Desktop\thesis_ef\evaluate\ground_truth_hybrid.csv'
df = pd.read_csv(gt)
synonym_dict = {row['Keyword']:row['Synonyms'].replace(' ','').split(',') for _,row in df.iterrows()}
ground_truth = {row['Keyword']:row['Rows Found'].strip("[]").split(", ") for _,row in df.iterrows()}

In [11]:
from elasticsearch import Elasticsearch
import pandas as pd
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

load_dotenv()
CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
API_KEY = os.getenv("ELASTIC_API_KEY")
es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
# Initialize the sentence transformer model for embeddings
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def hybrid_search_and_evaluate(query, synonym_dict, ground_truth, bm25_weight=0.6, vector_weight=0.4, 
                              index="thai_hybrid_search_ef", size=5):
    """
    ค้นหาใน Elasticsearch แบบ Hybrid (BM25 + Vector Search) พร้อมใช้ Synonym Matching 
    และคำนวณ Precision, Recall และ Average Precision
    
    Args:
        query: คำค้นหา
        synonym_dict: พจนานุกรม synonym ที่ใช้ขยายคำค้นหา
        ground_truth: พจนานุกรมของคำค้นหาและ ID ของเอกสารที่เกี่ยวข้อง
        bm25_weight: น้ำหนักของการค้นหาแบบ BM25 (0.0-1.0)
        vector_weight: น้ำหนักของการค้นหาแบบ Vector (0.0-1.0)
        index: ชื่อ index ใน Elasticsearch
        size: จำนวนผลลัพธ์ที่ต้องการ
        
    Returns:
        Dictionary ที่มีข้อมูลผลการค้นหาและการประเมินประสิทธิภาพ
    """
    try:
        if not query:
            return {"error": "No query provided."}

        # ขยาย Query ด้วย Synonym Dictionary
        expanded_queries = synonym_dict.get(query, [query])  # ใช้คำเดิมถ้าไม่มีใน Dictionary
        expanded_query_text = " ".join(expanded_queries)
        
        # สร้าง embedding สำหรับคำค้นหา
        query_vector = model.encode(query).tolist()
        
        # สร้าง query แบบ hybrid
        search_query = {
            "query": {
                "bool": {
                    "should": [
                        # BM25 search - จะใช้ synonyms จาก analyzer ที่กำหนดไว้
                        {
                            "multi_match": {
                                "query": expanded_query_text,
                                "fields": ["ชื่อ^3", "รายละเอียด^2"],
                                "type": "best_fields",
                                "operator": "or",
                                "boost": bm25_weight
                            }
                        },
                        # Vector search ใช้ knn_vector
                        {
                            "knn": {
                                "field": "text_vector",
                                "query_vector": query_vector,
                                "k": 10,
                                "num_candidates": 100,
                                "boost": vector_weight
                            }
                        }                        
                    ]
                }
            },
            "size": size
        }
        
        # ส่ง query ไปยัง Elasticsearch
        response = es.search(index=index, body=search_query)

        # ดึง ID ของเอกสารที่ได้รับ
        retrieved_ids = [hit["_id"] for hit in response['hits']['hits']]
        
        # ดึง ID ของเอกสารที่เกี่ยวข้องจาก ground truth
        relevant_ids = ground_truth.get(query, [])
        
        # คำนวณ True Positives, False Positives, False Negatives
        tp = len(set(retrieved_ids) & set(relevant_ids))  # True Positives
        fp = len(set(retrieved_ids) - set(relevant_ids))  # False Positives
        fn = len(set(relevant_ids) - set(retrieved_ids))  # False Negatives

        # คำนวณ Precision และ Recall
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # คำนวณ F1 Score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        # คำนวณ Average Precision (AP)
        num_relevant = 0
        precision_at_k = []
        
        for k, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_ids:
                num_relevant += 1
                precision_at_k.append(num_relevant / k)
        
        ap_score = sum(precision_at_k) / len(relevant_ids) if relevant_ids else 0
        
        # คำนวณ Mean Reciprocal Rank (MRR)
        reciprocal_rank = 0
        for rank, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break

        return {
            "query": query,
            "expanded_queries": expanded_queries,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "AP": ap_score,
            "reciprocal_rank": reciprocal_rank,
            "retrieved_ids": retrieved_ids,
            "relevant_ids": relevant_ids,
            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "bm25_weight": bm25_weight,
            "vector_weight": vector_weight
        }
    
    except Exception as e:
        return {"error": str(e)}

In [13]:
results_list = []

for key in ground_truth.keys():
    result = hybrid_search_and_evaluate(key, synonym_dict, ground_truth, bm25_weight=0.5, vector_weight=0.5)

    results_list.append({
        "Query": result["query"],
        "Expanded Queries": ", ".join(result["expanded_queries"]),
        "Precision": result["precision"],
        "Recall": result["recall"],
        "F1 Score": result["f1_score"],
        "AP": result["AP"],
        "Reciprocal Rank": result["reciprocal_rank"],
        "Retrieved IDs": ", ".join(result["retrieved_ids"]),
        "Relevant IDs": ", ".join(result["relevant_ids"]),
        "True Positives": result["true_positives"],
        "False Positives": result["false_positives"],
        "False Negatives": result["false_negatives"],
        "BM25 Weight": result["bm25_weight"],
        "Vector Weight": result["vector_weight"]
    })

# คำนวณ MAP (Mean Average Precision)
lst_ap = sum([value['AP'] for value in results_list])
map_score = lst_ap/len(results_list)

# คำนวณ MRR (Mean Reciprocal Rank)
lst_rr = sum([value['Reciprocal Rank'] for value in results_list])
mrr_score = lst_rr/len(results_list)

# คำนวณค่าเฉลี่ย Precision, Recall และ F1
avg_precision = sum([value['Precision'] for value in results_list])/len(results_list)
avg_recall = sum([value['Recall'] for value in results_list])/len(results_list)
avg_f1 = sum([value['F1 Score'] for value in results_list])/len(results_list)

# สร้าง DataFrame
df_results = pd.DataFrame(results_list)

# เพิ่มค่าเฉลี่ยให้กับทุกแถว
df_results['MAP'] = map_score
df_results['MRR'] = mrr_score
df_results['Avg Precision'] = avg_precision
df_results['Avg Recall'] = avg_recall
df_results['Avg F1'] = avg_f1

# บันทึกผลลัพธ์ลงไฟล์ CSV
df_results.to_csv("hybrid_evaluation_results.csv", index=False, encoding='utf-8-sig')

print(f"ผลการประเมิน:")
print(f"MAP: {map_score:.4f}")
print(f"MRR: {mrr_score:.4f}")
print(f"Avg Precision: {avg_precision:.4f}")
print(f"Avg Recall: {avg_recall:.4f}")
print(f"Avg F1: {avg_f1:.4f}")

ผลการประเมิน:
MAP: 0.1326
MRR: 0.2454
Avg Precision: 0.1628
Avg Recall: 0.2717
Avg F1: 0.1585
